In [1]:
import numpy as np
from matplotlib import pyplot as plt
from math import floor

from bff_paper_figures.extract_experiment_values import  get_true_transition_frequencies
from bff_sensitivity_calculations.sensitivity_functions import get_optimal_evolution_time_hf_agnostic_s, get_vdpr_and_ramsey_min_slopes_dbz_dsignal,get_vpdr_slope_dbz_dsignal_at_tau_opt, get_ramsey_slope_dbz_dsignal_at_tau_opt, optimal_to_avg_slope_ratio_dsignal_dlarmor, vpdr_and_ramsey_sensitivity_at_tau_opt, vpdr_and_ramsey_sensitivity_fitting
from bff_simulator.abstract_classes.abstract_ensemble import NVOrientation, NV14HyperfineField
from bff_simulator.constants import NVaxes_100, f_h
from bff_simulator.homogeneous_ensemble import HomogeneousEnsemble
from bff_simulator.liouvillian_solver import LiouvillianSolver
from bff_simulator.vector_manipulation import perpendicular_projection
from bff_simulator.offaxis_field_experiment_parameters import OffAxisFieldExperimentParametersFactory
from bff_paper_figures.shared_parameters import MW_DIRECTION, E_FIELD_VECTOR_V_PER_CM, RABI_FREQ_BASE_HZ, DETUNING_HZ, T2STAR_S, RAD_TO_DEGREE


DELTA_B_T = 1e-9
SEED = 29
SIG_STD_DEV = 1e-4
N_SAMPLES = 1000
N_SAMPLES_FIT = 100
S_TO_US = 1e6
S_TO_NS = 1e9

B_MAGNITUDE_T = 50e-6
B_THETA_START = 0
B_THETA_STOP = np.pi
B_THETA_N = 11

B_PHI_START = 0
B_PHI_STOP = 2 * np.pi
B_PHI_N_MAX = 21
B_PHI_N_MIN = 3

IDEAL_RABI_FREQUENCIES= np.array([RABI_FREQ_BASE_HZ * perpendicular_projection(MW_DIRECTION, NVaxis) for NVaxis in NVaxes_100])
INDEX_FOR_MI0 = 1
MW_PULSE_LENGTH_S = np.arange(0, 800e-9, 2.5e-9)  # np.linspace(0, 0.5e-6, 1001)
EVOLUTION_TIME_S = np.arange(0, 3e-6, 20e-9)  # p.linspace(0, 15e-6, 801)
S_TO_NS = 1e9

RABI_FREQ_BASE_HZ = 100e6
MAX_PULSE_DURATIONS = np.arange(50e-9, 800e-9, 50e-9)
MAX_EVOLUTION_TIMES_S = np.arange(400e-9, 4e-6, 200e-9)

use_hyperfine = False
seed = 29
orientation = NVOrientation.B
rabi_window_name = "blackman"

def get_all_sensitivities_vs_b_angle(orientation, use_hyperfine, seed, rabi_window_name):

    rabi_freq_hz = IDEAL_RABI_FREQUENCIES[orientation]
    exp_param_factory = OffAxisFieldExperimentParametersFactory()
    exp_param_factory.set_base_rabi_frequency(RABI_FREQ_BASE_HZ)
    exp_param_factory.set_mw_direction(MW_DIRECTION)
    exp_param_factory.set_e_field_v_per_m(E_FIELD_VECTOR_V_PER_CM)
    exp_param_factory.set_detuning(DETUNING_HZ)
    exp_param_factory.set_mw_pulse_lengths(MW_PULSE_LENGTH_S)
    exp_param_factory.set_evolution_times(EVOLUTION_TIME_S)

    nv_ensemble = HomogeneousEnsemble()
    nv_ensemble.t2_star_s = T2STAR_S
    if use_hyperfine:
        nv_ensemble.add_n14_triplet(orientation)
    else:
        nv_ensemble.add_nv_single_species(orientation, NV14HyperfineField.N14_0)

    off_axis_solver = LiouvillianSolver()

    # Calculate the pi pulse time
    t_pi_s = 1/(2*rabi_freq_hz)
    print(f"Pi pulse duration: {t_pi_s*S_TO_NS} ns")

    rng = np.random.default_rng(seed)

    theta_values = []
    phi_values = []
    vpdr_sensitivities = []
    ramsey_sensitivities = []
    slope_ratios = []
    vpdr_fit_sensitivities = []
    ramsey_fit_sensitivities =[]

    for theta in np.linspace(B_THETA_START, B_THETA_STOP, B_THETA_N):
        print(f"Theta = {theta*RAD_TO_DEGREE:.1f} degrees")
        b_phi_n = max(B_PHI_N_MIN, floor(np.sin(theta) * B_PHI_N_MAX))
        for phi in np.linspace(B_PHI_START, B_PHI_STOP, b_phi_n):

            b_field_vector_t = B_MAGNITUDE_T * np.array([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)])
            exp_param_factory.set_b_field_vector(b_field_vector_t)
            
            # Extract the true mi = 0 larmor precession frequency and use it to find the optimal free precession time
            larmor_freqs_all_axes_hz, _ = get_true_transition_frequencies(exp_param_factory.get_experiment_parameters())
            double_larmor_mi0_hz = larmor_freqs_all_axes_hz[orientation][INDEX_FOR_MI0] # double quantum larmor frequency
            print(f"Larmor frequency: {double_larmor_mi0_hz/2 * 1e-6:.02} MHz")
            optimal_evolution_time_s = get_optimal_evolution_time_hf_agnostic_s(double_larmor_mi0_hz, T2STAR_S, use_hyperfine, f_h, EVOLUTION_TIME_S, do_plot=False)

            ##################### Determine the optimal-time sensitivities #############################
            # Find the signal response to magnetic field at the best time (which may not be quite the theoretical optimum due to precession during the MW pulses)
            # min_dbz_dsignal_vpdr, min_dbz_dsignal_ramsey, tau_opt_vpdr_s, tau_opt_ramsey_s = get_vdpr_and_ramsey_min_slopes_dbz_dsignal(exp_param_factory, nv_ensemble, off_axis_solver, rabi_window_name, optimal_evolution_time_s, t_pi_s, b_field_vector_t, b_field_vector_t*(1+DELTA_B_T/B_MAGNITUDE_T), orientation, INDEX_FOR_MI0, do_plots = False)
            # print(f"Optimal free evolution times for VPDR: {tau_opt_vpdr_s*S_TO_US:.03f} and Ramsey: {tau_opt_ramsey_s*S_TO_US:.03f} vs theory: {optimal_evolution_time_s*S_TO_US:.03f}")
            if optimal_evolution_time_s > 0:
                min_dbz_dsignal_vpdr = np.abs(get_vpdr_slope_dbz_dsignal_at_tau_opt(optimal_evolution_time_s, exp_param_factory, nv_ensemble, off_axis_solver, rabi_window_name,  b_field_vector_t, b_field_vector_t*(1+DELTA_B_T/B_MAGNITUDE_T), orientation, INDEX_FOR_MI0))
                min_dbz_dsignal_ramsey = np.abs(get_ramsey_slope_dbz_dsignal_at_tau_opt(optimal_evolution_time_s, exp_param_factory, nv_ensemble, off_axis_solver, t_pi_s,   b_field_vector_t, b_field_vector_t*(1+DELTA_B_T/B_MAGNITUDE_T),orientation, INDEX_FOR_MI0))
            else:
                min_dbz_dsignal_vpdr, min_dbz_dsignal_ramsey, tau_opt_vpdr_s, tau_opt_ramsey_s = get_vdpr_and_ramsey_min_slopes_dbz_dsignal(exp_param_factory, nv_ensemble, off_axis_solver, rabi_window_name, optimal_evolution_time_s, t_pi_s, b_field_vector_t, b_field_vector_t*(1+DELTA_B_T/B_MAGNITUDE_T), orientation, INDEX_FOR_MI0, do_plots = False)

            vpdr_sensitivity, ramsey_sensitivity = vpdr_and_ramsey_sensitivity_at_tau_opt(exp_param_factory.get_experiment_parameters(), orientation, rabi_window_name, min_dbz_dsignal_vpdr, min_dbz_dsignal_ramsey, rng, N_SAMPLES, SIG_STD_DEV)
            
            #################### Determine the sensitivity from fitting #################################
            vpdr_fit_sensitivity, ramsey_fit_sensitivity = vpdr_and_ramsey_sensitivity_fitting(exp_param_factory, nv_ensemble, off_axis_solver, t_pi_s, orientation, rabi_window_name, use_hyperfine, double_larmor_mi0_hz, rng, N_SAMPLES_FIT, SIG_STD_DEV)

            ################### Calculate slope ratios ###################################
            theory_slope_ratio = optimal_to_avg_slope_ratio_dsignal_dlarmor(EVOLUTION_TIME_S, optimal_evolution_time_s, double_larmor_mi0_hz, f_h, T2STAR_S, use_hyperfine)

            vpdr_sensitivities.append(vpdr_sensitivity)
            ramsey_sensitivities.append(ramsey_sensitivity)
            theta_values.append(theta)
            phi_values.append(phi)
            vpdr_fit_sensitivities.append(vpdr_fit_sensitivity)
            ramsey_fit_sensitivities.append(ramsey_fit_sensitivity)
            slope_ratios.append(theory_slope_ratio)
    return theta_values, phi_values, vpdr_fit_sensitivities, vpdr_sensitivities, ramsey_fit_sensitivities, ramsey_sensitivities, slope_ratios
    

In [ ]:
theta_values, phi_values, vpdr_fit_sensitivities_a, vpdr_sensitivities_a, ramsey_fit_sensitivities_a, ramsey_sensitivities_a, slope_ratios_a = get_all_sensitivities_vs_b_angle(NVOrientation.A, use_hyperfine=True, seed=SEED, rabi_window_name="blackman")
theta_values, phi_values, vpdr_fit_sensitivities_b, vpdr_sensitivities_b, ramsey_fit_sensitivities_b, ramsey_sensitivities_b, slope_ratios_b = get_all_sensitivities_vs_b_angle(NVOrientation.B, use_hyperfine=True, seed=SEED, rabi_window_name="blackman")
theta_values, phi_values, vpdr_fit_sensitivities_c, vpdr_sensitivities_c, ramsey_fit_sensitivities_c, ramsey_sensitivities_c, slope_ratios_c = get_all_sensitivities_vs_b_angle(NVOrientation.C, use_hyperfine=True, seed=SEED, rabi_window_name="blackman")
theta_values, phi_values, vpdr_fit_sensitivities_d, vpdr_sensitivities_d, ramsey_fit_sensitivities_d, ramsey_sensitivities_d, slope_ratios_d = get_all_sensitivities_vs_b_angle(NVOrientation.D, use_hyperfine=True, seed=SEED, rabi_window_name="blackman")

Pi pulse duration: 7.534157191110836 ns
Theta = 0.0 degrees
Larmor frequency: 0.81 MHz
Larmor frequency: 0.81 MHz
Larmor frequency: 0.81 MHz
Theta = 18.0 degrees
Larmor frequency: 1.0 MHz
Larmor frequency: 1.1 MHz
Larmor frequency: 0.71 MHz
Larmor frequency: 0.42 MHz
Larmor frequency: 0.61 MHz
Larmor frequency: 1.0 MHz
Theta = 36.0 degrees
Larmor frequency: 1.1 MHz
Larmor frequency: 1.3 MHz
Larmor frequency: 1.3 MHz
Larmor frequency: 1.1 MHz


In [ ]:
plt.tripcolor(np.array(phi_values) * RAD_TO_DEGREE, np.array(theta_values) * RAD_TO_DEGREE, np.array(vpdr_sensitivities_a)/np.array(ramsey_sensitivities_a))
plt.colorbar()
plt.show()
plt.tripcolor(np.array(phi_values) * RAD_TO_DEGREE, np.array(theta_values) * RAD_TO_DEGREE, np.array(vpdr_sensitivities_b)/np.array(ramsey_sensitivities_b))
plt.colorbar()
plt.show()
plt.tripcolor(np.array(phi_values) * RAD_TO_DEGREE, np.array(theta_values) * RAD_TO_DEGREE, np.array(vpdr_sensitivities_c)/np.array(ramsey_sensitivities_c))
plt.colorbar()
plt.show()
plt.tripcolor(np.array(phi_values) * RAD_TO_DEGREE, np.array(theta_values) * RAD_TO_DEGREE, np.array(vpdr_sensitivities_d)/np.array(ramsey_sensitivities_d))
plt.colorbar()
plt.show()

In [ ]:
plt.tripcolor(np.array(phi_values) * RAD_TO_DEGREE, np.array(theta_values) * RAD_TO_DEGREE, np.array(vpdr_fit_sensitivities_a)/np.array(vpdr_sensitivities_a))
plt.colorbar()
plt.show()
plt.tripcolor(np.array(phi_values) * RAD_TO_DEGREE, np.array(theta_values) * RAD_TO_DEGREE, np.array(vpdr_fit_sensitivities_b)/np.array(vpdr_sensitivities_b))
plt.colorbar()
plt.show()
plt.tripcolor(np.array(phi_values) * RAD_TO_DEGREE, np.array(theta_values) * RAD_TO_DEGREE, np.array(vpdr_fit_sensitivities_c)/np.array(vpdr_sensitivities_c))
plt.colorbar()
plt.show()
plt.tripcolor(np.array(phi_values) * RAD_TO_DEGREE, np.array(theta_values) * RAD_TO_DEGREE, np.array(vpdr_fit_sensitivities_d)/np.array(vpdr_sensitivities_d))
plt.colorbar()
plt.show()